In [98]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

from collections import OrderedDict

In [ ]:
H = int
W = int
C = int
class DarkNet53ResidualBlock(nn.Module):
    def __init__(self, channel_dims: list[C]):
        super().__init__()
        self.lrelu = nn.LeakyReLU()
        self.conv1 = nn.Conv2d(channel_dims[0], channel_dims[1], 1)
        self.bnorm = nn.BatchNorm2d(channel_dims[1])

        self.conv2 = nn.Conv2d(channel_dims[1], channel_dims[2], 3, padding=1)
        self.bnorm = nn.BatchNorm2d(channel_dims[2])
        
    
    def forward(self, input):
        x = self.lrelu(self.conv1(input))
        x = self.lrelu(self.conv2(x))
        return x + input
    

class DarkNet53(nn.Module):
    def __init__(self):
        super().__init__()

        self.lrelu = nn.LeakyReLU()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, 2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, 2, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, 2, padding=1)
        self.conv5 = nn.Conv2d(256, 512, 3, 2, padding=1)
        self.conv6 = nn.Conv2d(512, 1024, 3, 2, padding=1)

        self.darknet_block1 = DarkNet53ResidualBlock([64, 32, 64])
        self.darknet_block2 = nn.ModuleList([DarkNet53ResidualBlock([128, 64, 128]) for _ in range(2)])
        self.darknet_block3 = nn.ModuleList([DarkNet53ResidualBlock([256, 128, 256]) for _ in range(8)])
        self.darknet_block4 = nn.ModuleList([DarkNet53ResidualBlock([512, 256, 512]) for _ in range(8)])
        self.darknet_block5 = nn.ModuleList([DarkNet53ResidualBlock([1024, 512, 1024]) for _ in range(4)])

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.dense = nn.Linear(1024, 1000)
        self.softmax = nn.Softmax()

    def forward(self, input):
        x = self.lrelu(self.conv1(input))
        x = self.lrelu(self.conv2(x))

        x = self.darknet_block1(x)
        x = self.lrelu(self.conv3(x))

        for darknet_block in self.darknet_block2:
            x = darknet_block(x)
        x = self.lrelu(self.conv4(x))

        for darknet_block in self.darknet_block3:
            x = darknet_block(x)
        x = self.lrelu(self.conv5(x))

        for darknet_block in self.darknet_block4:
            x = darknet_block(x)
        x = self.lrelu(self.conv6(x))

        for darknet_block in self.darknet_block5:
            x = darknet_block(x)

        x = torch.flatten(self.avgpool(x),1)
        print(x.size)
        x = self.dense(x)
        x = self.softmax(x)
        return x
    


In [106]:
test = DarkNet53()
sample = torch.rand((1, 3, 256, 256))
test(sample).size()

<built-in method size of Tensor object at 0x75e2085304b0>


/home/shashankg/packages/YOLOX/yolo/lib/python3.12/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


torch.Size([1, 1000])

In [95]:
print([DarkNet53ResidualBlock([128, 64, 128])]*2)

[DarkNet53ResidualBlock(
  (lrelu): LeakyReLU(negative_slope=0.01)
  (conv1): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
), DarkNet53ResidualBlock(
  (lrelu): LeakyReLU(negative_slope=0.01)
  (conv1): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)]


In [ ]:
Ordered